# Combining Reranking and Hybrid Search to improve data Retrieval

In [1]:
from dotenv import load_dotenv
load_dotenv("../../.env")

import openai
import pandas as pd

from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, SparseVectorParams, Modifier, PayloadSchemaType, PointStruct, Document, Prefetch, RrfQuery, Rrf

from sentence_transformers import CrossEncoder

In [2]:
qdrant_client = QdrantClient(url="http://localhost:6333")

### Embed text

In [3]:
def get_embeddings_batch(text_list, model="text-embedding-3-small", batch_size=100):
    if len(text_list) <= batch_size:
        response = openai.embeddings.create(input=text_list, model=model)
        return [embedding.embedding for embedding in response.data]

    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i+batch_size]
        response = openai.embeddings.create(input=batch, model=model)
        all_embeddings.extend(
            [embedding.embedding for embedding in response.data]
        )

        print(f"Batch #{counter}: {counter * batch_size} / {len(text_list)}")
        counter += 1

    return all_embeddings

In [4]:
def retrieve_data_hybrid(query, k=10):
    query_embedding = get_embeddings_batch(query)[0]

    #hybrid retrieval
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-hybrid-search",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                limit=k
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=k
            )
        ],
        query=RrfQuery(rrf=Rrf(weights=[0.4, 0.6])),
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


In [30]:
QUERY = "What are some available speakers?"
results = retrieve_data_hybrid(QUERY, k=20)

In [31]:
results

{'retrieved_context_ids': ['B0C996WY16',
  'B0B5LW6277',
  'B0C8S6BBY9',
  'B0C4NJPN4Q',
  'B09WCT9S1R',
  'B0BBF2VC6X',
  'B0CH8DRD6K',
  'B09TFM1SFQ',
  'B0BRV544MV',
  'B0CC4HBS85',
  'B09X9838WY',
  'B0CF57H28T',
  'B0BM657X74',
  'B0BG6TMXDD',
  'B09PTX6461',
  'B0B8ZMQ53J',
  'B0CFHWF326',
  'B09KQP2H7N',
  'B09P4QW5Y2',
  'B0BRJS644Z'],
 'retrieved_context': ['Raymate Bluetooth Speakers, HiFi Stereo Sound with DSP, 30W IPX7 Waterproof Speaker Wireless Bluetooth-V5.0, 1000mins Playtime, Portable Speaker for Home, Outdoor, Party 🎶HiFi Sound: With proven audio processing DSP chip technology, 30W dual speaker drivers and a more powerful amplifier module, the portable speakers delivers even, Balanced Sound Without Distortion. 🧱Integrated structure: With IPX7 Waterproof Speakers protection against rain, dust, snow and splashes, you can enjoy music in the pool, beach, park, bathroom and beyond.Comes with a dirt and corrosion resistant portable silicone protective sleeve. ✨Simple And St

### Reranking the top-k results

In [32]:
# detect ideal accelerator, if available
import torch as T
def get_device():
    if T.cuda.is_available():
        return "cuda"
    elif hasattr(T.backends, "mps") and T.backends.mps.is_available():
        # Check if MPS is built and available on Mac
        return "mps"
    return "cpu"

get_device()

'mps'

In [33]:
model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device=get_device())

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6869.47it/s]


In [34]:
rerank_scores = model.rank(QUERY, results["retrieved_context"])

In [35]:
rerank_scores

[{'corpus_id': 0, 'score': np.float32(-3.809554)},
 {'corpus_id': 3, 'score': np.float32(-5.509267)},
 {'corpus_id': 9, 'score': np.float32(-6.336402)},
 {'corpus_id': 6, 'score': np.float32(-6.8404274)},
 {'corpus_id': 10, 'score': np.float32(-7.1433)},
 {'corpus_id': 2, 'score': np.float32(-7.2605853)},
 {'corpus_id': 5, 'score': np.float32(-8.159249)},
 {'corpus_id': 12, 'score': np.float32(-8.323565)},
 {'corpus_id': 17, 'score': np.float32(-8.514812)},
 {'corpus_id': 7, 'score': np.float32(-8.570568)},
 {'corpus_id': 16, 'score': np.float32(-8.761023)},
 {'corpus_id': 1, 'score': np.float32(-8.911423)},
 {'corpus_id': 18, 'score': np.float32(-8.983621)},
 {'corpus_id': 8, 'score': np.float32(-9.162359)},
 {'corpus_id': 15, 'score': np.float32(-9.310589)},
 {'corpus_id': 14, 'score': np.float32(-9.984316)},
 {'corpus_id': 13, 'score': np.float32(-10.09494)},
 {'corpus_id': 11, 'score': np.float32(-10.65045)},
 {'corpus_id': 19, 'score': np.float32(-11.023853)},
 {'corpus_id': 4, 's

In [36]:
reranked_results = [results["retrieved_context"][rerank["corpus_id"]] for rerank in rerank_scores]

In [37]:
reranked_results

['Raymate Bluetooth Speakers, HiFi Stereo Sound with DSP, 30W IPX7 Waterproof Speaker Wireless Bluetooth-V5.0, 1000mins Playtime, Portable Speaker for Home, Outdoor, Party 🎶HiFi Sound: With proven audio processing DSP chip technology, 30W dual speaker drivers and a more powerful amplifier module, the portable speakers delivers even, Balanced Sound Without Distortion. 🧱Integrated structure: With IPX7 Waterproof Speakers protection against rain, dust, snow and splashes, you can enjoy music in the pool, beach, park, bathroom and beyond.Comes with a dirt and corrosion resistant portable silicone protective sleeve. ✨Simple And Stylish: This is a speaker with a golden ratio rectangular body, combined with the purest colors, perfect for a variety of venues. 🔋Extended Battery Life: 3600mAh high-capacity battery provides you with more than 1000mins of play time.This speaker uses a more stable charging port - type C port, which greatly saves your waiting time. Note: We provide a 180 day warranty

>Findings: It seems like the reranker correctly identified, that some items that do have speakers like a TV but are not primarily speakers were moved further down the list. While some items where incorrectly classified as speakers, like audio cables.